# Referenzquelle NIBRS: die gemeinsame Deliktsprache

Diese Quelle liefert keine Zeilen, sondern **Bedeutung**. Chicago und Detroit
zählen dieselben Taten unter verschiedenen Schlüsseln; ohne eine gemeinsame
Klassifikation ist jeder Vergleich eine Behauptung.

**Pipeline-Stufe:** Referenzdokumentation und Qualitätssicherung der
Referenzdaten. Es gibt keine Ingestion — die Zuordnung liegt als gepflegte
Tabellen in `config/` und wird in `silver/crime_offense` angewandt.

| | |
| --- | --- |
| Herausgeber | FBI, gemeinsam mit dem Bureau of Justice Statistics |
| Rolle | nationaler Standard der Kriminalitätserfassung der USA seit Januar 2021 |
| Vorgänger | Summary Reporting System (SRS), zum 1.1.2021 eingestellt |
| Im Projekt | Zielsprache für Chicagos IUCR und Detroits MICR |

**Was dieses Notebook zeigt.** Referenzdaten sind der blinde Fleck der
Datenqualität: Sie sind klein, ändern sich selten und werden deshalb selten
geprüft — aber ein Fehler in ihnen verfälscht *jede* Auswertung, die über sie
läuft, und zwar unauffällig. Dieses Notebook prüft sie.

Drei Fallen stecken darin, und alle drei sind uns tatsächlich passiert:

1. **Gleicher Name, andere Bedeutung** — Chicagos `fbi_code` heisst nach dem
   FBI, ist aber SRS und nicht NIBRS (Abschnitt 2).
2. **Gleiche Form, anderer Schlüssel** — die erste Fassung des
   Chicago-Crosswalks entstand aus einem Join über ein vierstelliges Kürzel,
   das in zwei Systemen Verschiedenes bedeutet. Das Ergebnis war formgültig und
   falsch (Abschnitt 4).
3. **Ein Feld, zwei Codelisten** — Detroits `state_offense_code` mischt MICR
   File Classes und Arrest Charges, die im vierstelligen Raum kollidieren
   (Abschnitt 6.1).

Keine dieser Fallen fällt bei einer Schemaprüfung auf. Gefunden hat sie jedes
Mal der Abgleich der **Bedeutung** — der Deliktbeschreibungen, die beide Seiten
mitliefern. Die Quellenverweise stehen am Ende in Abschnitt 8.

## 1. Was NIBRS ist

**NIBRS** steht für *National Incident-Based Reporting System*. Es ist seit
Januar 2021 der nationale Standard für die Meldung von Kriminalitätsdaten in
den USA und wird von FBI und BJS gemeinsam betrieben.

Der entscheidende Unterschied zum Vorgänger liegt im Namen: **incident-based**.
Das SRS zählte je Vorfall nur die schwerste Straftat — die sogenannte
Hierarchieregel. NIBRS erfasst mehrere Delikte je Vorfall, dazu Angaben zu
Opfern, Tatverdächtigen, Waffen und Sachschaden.

Für dieses Projekt ist das mehr als eine Fussnote: Ein Datensatz mit
Hierarchieregel zählt weniger Delikte als einer ohne, **bei identischer
Kriminalität**. Wer zwei Quellen vergleicht, die diese Regel unterschiedlich
anwenden, vergleicht Erfassungsverfahren und nicht Taten.

Genau deshalb führt das Projekt in Silver `event_key` und `offense_key`
getrennt: Ein Ereignis kann mehrere Delikte tragen.

## 2. Die Falle: Chicagos `fbi_code` ist **nicht** NIBRS

Chicagos Falldaten führen eine Spalte `fbi_code`. Der Name legt nahe, dass dort
der FBI-Standard steht — also NIBRS. Das stimmt nicht: Es ist der **ältere
UCR-/SRS-Schlüssel**.

Die beiden Codesysteme sehen ähnlich aus, sind aber verschiedene Namensräume.
Die nächste Zelle prüft das an den Daten, statt es zu behaupten.

In [ ]:
from pathlib import Path
import csv
import collections

import duckdb

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'config').exists():
    raise RuntimeError('Projekt-Root mit config/ nicht gefunden')

IUCR_MAPPING = PROJECT_ROOT / 'config' / 'iucr_nibrs_mapping.csv'
MICR_MAPPING = PROJECT_ROOT / 'config' / 'micr_nibrs_mapping.csv'
NIBRS_CODES = PROJECT_ROOT / 'config' / 'nibrs_offense_codes_2025.csv'

iucr_map = list(csv.DictReader(IUCR_MAPPING.open(encoding='utf-8')))
micr_map = list(csv.DictReader(MICR_MAPPING.open(encoding='utf-8')))
nibrs_codes = {z['nibrs_code']: z for z in csv.DictReader(NIBRS_CODES.open(encoding='utf-8'))}

con = duckdb.connect()
CHICAGO = f"read_parquet('{PROJECT_ROOT}/data/bronze/city=chicago/**/*.parquet')"
# Detroit schreibt Bronze als Parquet in Laufverzeichnissen. Der frueher
# gelesene NDJSON-Bestand ist geloescht: Sein Jahresfenster war in UTC gezogen
# statt in Detroiter Ortszeit und enthielt deshalb Faelle aus 2023.
_detroit = sorted(
    p for p in (PROJECT_ROOT / 'data' / 'bronze' / 'detroit').glob(
        '**/run_*/part-0001.parquet')
    if not any(teil.startswith('.') for teil in p.parts)
)
if not _detroit:
    raise RuntimeError(
        'Kein Detroit-Bronze-Lauf. Zuerst: '
        'python -m detroit_pipeline.extract_detroit --mode full --years 2024'
    )
DETROIT = f"read_parquet({[str(x) for x in _detroit]})"

# Die Silver-Tabelle, in der beide Crosswalks angewandt sind. Sie entsteht mit
# python -m silver_pipeline.build_crime_offense und fehlt, solange das nicht lief.
_offense = PROJECT_ROOT / 'data' / 'silver' / 'dataset=crime_offense' / 'part-0001.parquet'
OFFENSE = f"read_parquet('{_offense}')" if _offense.exists() else None

print('IUCR-Mapping   :', len(iucr_map), 'Zeilen')
print('MICR-Mapping   :', len(micr_map), 'Zeilen')
print('NIBRS-Codeliste:', len(nibrs_codes), 'amtliche Codes')
print('Chicago-Bronze :', f"{con.execute(f'SELECT count(*) FROM {CHICAGO}').fetchone()[0]:,}", 'Zeilen')
print('Detroit-Bronze :', f"{con.execute(f'SELECT count(*) FROM {DETROIT}').fetchone()[0]:,}",
      'Zeilen (Parquet)')
print('silver/crime_offense:',
      f"{con.execute(f'SELECT count(*) FROM {OFFENSE}').fetchone()[0]:,} Zeilen" if OFFENSE
      else 'fehlt - zuerst build_crime_offense laufen lassen')

### 2.1 Zwei Namensräume, keine Überschneidung

In [ ]:
fbi_codes = {z[0] for z in con.execute(
    f'SELECT DISTINCT fbi_code FROM {CHICAGO} WHERE fbi_code IS NOT NULL'
).fetchall()}
ibr_codes = {z['ibr_code'] for z in iucr_map if z['ibr_code'].strip()}

print(f'fbi_code in den Falldaten : {len(fbi_codes):>3} verschiedene')
print('   ', ', '.join(sorted(fbi_codes)))
print()
print(f'ibr_code im Mapping       : {len(ibr_codes):>3} verschiedene')
print('   ', ', '.join(sorted(ibr_codes)))
print()
gemeinsam = fbi_codes & ibr_codes
print('gemeinsame Codes:', sorted(gemeinsam) if gemeinsam else 'KEINE')
print()
print('-> zwei getrennte Codesysteme, nicht zwei Schreibweisen desselben')

### 2.2 Dieselbe Zeile, zwei Schlüssel

Beide Systeme beschreiben oft dasselbe — aber mit anderem Code und anderer
Feinheit. `08B` ist der SRS-Code für einfache Körperverletzung, `13B` der
NIBRS-Code dafür.

In [ ]:
zu_ibr = {z['iucr']: (z['ibr_code'], z['ibr_category']) for z in iucr_map}

print(f"{'IUCR':<6}{'primary_type':<24}{'fbi_code':<10}{'ibr_code':<10}ibr_category")
for iucr, typ, fbi, n in con.execute(f'''
    SELECT iucr, any_value(primary_type), any_value(fbi_code), count(*)
    FROM {CHICAGO} GROUP BY iucr ORDER BY count(*) DESC LIMIT 8
''').fetchall():
    code, kategorie = zu_ibr.get(iucr, ('-', '-'))
    print(f'{iucr:<6}{typ[:22]:<24}{fbi:<10}{code:<10}{kategorie[:30]}')

**Warum das zählt.** Die vergleichbare Kategorie des Projekts wird aus
`fbi_code` gebildet, also aus **SRS**, nicht aus NIBRS — siehe Abschnitt 3.2.
Das NIBRS-Mapping existiert daneben und ist feiner (51 gegen 26 Codes).

Das ist kein Fehler, aber es gehört gewusst: Wer sagt „wir harmonisieren über
NIBRS", beschreibt derzeit die feinere Tabelle, nicht den Weg, den die
Kategorien tatsächlich nehmen.

## 3. Die Referenz- und Mapping-Tabellen des Projekts

Sie tun Verschiedenes und werden leicht verwechselt.

| Datei | Von | Nach | Ebene |
| --- | --- | --- | --- |
| `config/nibrs_offense_codes_2025.csv` | — | die 81 amtlichen NIBRS-Codes | Zielliste, kein Mapping |
| `config/iucr_nibrs_mapping.csv` | Chicagos `iucr` | NIBRS `ibr_code` | fein, 54 Codes |
| `config/micr_nibrs_mapping.csv` | Detroits `state_offense_code` | NIBRS `nibrs_code` | fein, 63 Codes |

Die erste Datei ist der Grund, dass die anderen überhaupt prüfbar sind: Ohne
amtliche Zielliste kann niemand sagen, ob ein Mapping auf einen Code zeigt, den
es gibt. Sie stammt aus Anhang A der *NIBRS Technical Specification 2025* und
liegt als PDF unter `docs/`.

### 3.1 `iucr_nibrs_mapping.csv` — Herkunft nachgewiesen

Die Tabelle hat zehn Spalten. **Fünf davon stammen unverändert aus dem
offiziellen Datensatz des Chicago Data Portal**, die anderen fünf sind unsere
Ergänzung. Die nächste Zelle prüft das gegen die Quelle — sie braucht Netz.

In [ ]:
import requests

BASIS_SPALTEN = {'iucr', 'primary_description', 'secondary_description',
                 'index_code', 'active'}
PORTAL = 'https://data.cityofchicago.org/resource/c7ck-438e.json'

try:
    sitzung = requests.Session()
    sitzung.headers['User-Agent'] = 'crime-data-gruppe3/0.1'
    probe = sitzung.get(PORTAL, params={'$limit': 1}, timeout=60)
    probe.raise_for_status()
    anzahl = sitzung.get(PORTAL, params={'$select': 'count(*)'}, timeout=60).json()
    portal_spalten = set(probe.json()[0])
    portal_zeilen = int(anzahl[0]['count'])

    print('Quelle  :', PORTAL)
    print('Zeilen  :', f'im Portal {portal_zeilen}, bei uns {len(iucr_map)}',
          '-> gleich' if portal_zeilen == len(iucr_map) else '-> ABWEICHUNG')
    print('Spalten :', sorted(portal_spalten))
    print('  identisch mit unseren Basisspalten:',
          portal_spalten == BASIS_SPALTEN)
    print()
    print('unsere Ergaenzung:', sorted(set(iucr_map[0]) - portal_spalten))
except requests.RequestException as fehler:
    print('Portal nicht erreichbar:', fehler)
    print('unsere Spalten:', sorted(iucr_map[0]))

### 3.2 Die vergleichbare Ebene: NIBRS-Deliktfamilien

Ein einzelner NIBRS-Code ist für eine Auswertung zu fein — 54 Codes in Chicago,
63 in Detroit, viele davon mit einer Handvoll Fälle. Gebraucht wird eine
gröbere Ebene, auf der sich Zellen füllen.

NIBRS liefert sie selbst mit: Die **ersten beiden Zeichen** des Codes bilden die
Deliktfamilie. `13A`, `13B` und `13C` gehören zu `13` Körperverletzung, `26A`
bis `26G` zu `26` Betrug. Das ist keine eigene Erfindung, sondern die Gliederung
des Standards.

> **Zuvor stand hier `config/crime_category_mapping.csv`** — eine handgepflegte
> Liste, die Chicagos `fbi_code` (SRS) und Detroits `offense_category` auf 22
> gemeinsame Kategorien abbildete. Sie ist entfernt. Seit beide Städte über
> NIBRS laufen, wäre sie eine zweite Übersetzung neben der belegten, und sie
> hätte gepflegt werden müssen.

In [ ]:
familien = con.execute(f'''
    SELECT substr(nibrs_code, 1, 2) AS familie,
           count(*) FILTER (city = 'chicago') AS chicago,
           count(*) FILTER (city = 'detroit') AS detroit,
           any_value(nibrs_description) AS beispiel
    FROM {OFFENSE} WHERE nibrs_code IS NOT NULL
    GROUP BY 1 ORDER BY 1
''').fetchall() if OFFENSE else []

if not familien:
    print('silver/crime_offense fehlt - zuerst build_crime_offense laufen lassen')
else:
    beide = [f for f in familien if f[1] > 0 and f[2] > 0]
    print(f'{"Familie":<9}{"Chicago":>9}{"Detroit":>9}   Beispiel')
    for fam, chi, det, bsp in familien:
        marke = '' if chi and det else '   <- nur eine Stadt'
        print(f'{fam:<9}{chi:>9,}{det:>9,}   {bsp[:34]}{marke}')
    print()
    print(f'Familien insgesamt        : {len(familien)}')
    print(f'in BEIDEN Staedten belegt : {len(beide)}')
    gedeckt = sum(f[1] + f[2] for f in beide)
    gesamt = sum(f[1] + f[2] for f in familien)
    print(f'davon gedeckte Faelle     : {gedeckt:,} von {gesamt:,} = {gedeckt/gesamt:.2%}')

**20 der 25 Familien sind in beiden Städten belegt** — nur sie tragen eine
vergleichbare Zelle, und sie decken über 99 % der Fälle ab. Die fünf übrigen
kommen jeweils nur in einer Stadt vor: `27` Embezzlement, `37` Pornografie,
`51` Bestechung, `64` Menschenhandel und `72` Tierquälerei, alle in Chicago.

**Es liegt nicht am Crosswalk.** Die MICR-Tabelle führt beide Codes — `2700`
Embezzlement unter File Class `27000`, `7200` Animal Cruelty —, beide mit
`seen_in_detroit_2024 = false`. Detroit hat im Jahrgang 2024 also keinen Fall
darunter gemeldet. Ob das Meldepraxis ist oder tatsächlich null Fälle, sagen
die Daten nicht.

**Das ist die Ebene, auf der Gold aggregiert** (Konzept 5.2): `geoid` × Monat ×
Deliktfamilie.

### 3.3 Zeigen die Crosswalks überhaupt auf gültige Codes?

Ein Crosswalk kann formal tadellos sein und trotzdem ins Leere zeigen. Solange
niemand prüft, ob die Zielcodes existieren, läuft ein Tippfehler oder ein
zurückgezogener Code unbemerkt durch beide Tabellen.

`config/nibrs_offense_codes_2025.csv` hält deshalb die **81 Codes aus Anhang A
der NIBRS Technical Specification 2025** — die Zielliste, gegen die sich beide
Crosswalks prüfen lassen. Die nächste Zelle tut genau das.

In [ ]:
def pruefe(name, zeilen, spalte):
    codes = collections.Counter(z[spalte] for z in zeilen if z[spalte])
    unbekannt = sorted(set(codes) - set(nibrs_codes))
    bund = sorted(c for c in codes if nibrs_codes.get(c, {}).get('federal_only') == 'true')
    print(f'{name}: {len(codes)} verschiedene Zielcodes, {sum(codes.values())} zugeordnete Zeilen')
    print(f'  nicht in der Fassung 2025 : {unbekannt or "keiner"}')
    print(f'  nur von Bundesbehoerden   : {bund or "keiner"}')
    return unbekannt

weg_chicago = pruefe('Chicago (IUCR)', iucr_map, 'ibr_code')
print()
weg_detroit = pruefe('Detroit (MICR)', micr_map, 'nibrs_code')

print()
print('Die zurueckgezogenen Codes im Detail:')
for c in weg_detroit:
    treffer = [z for z in micr_map if z['nibrs_code'] == c]
    in_daten = sum(1 for z in treffer if z['seen_in_detroit_2024'] == 'true')
    print(f"  {c}  {treffer[0]['nibrs_description']:<26}"
          f"{len(treffer)} MICR-Code(s), davon in Detroit 2024: {in_daten}")

**Chicago ist sauber, Detroit nicht ganz.** Vier NIBRS-Codes hat das FBI
zurückgezogen: `90A` Bad Checks, `90E` Drunkenness, `90H` Peeping Tom und `90I`
Runaway. Sie stehen weiter im Crosswalk — die MICR-Quellen von 2009 und 2024
führen sie —, tragen aber eine Notiz.

Bemerkenswert: Bei `90A` und `90I` deckt sich das mit einem Vermerk, der schon
vorher in der Tabelle stand (*„nur in Appendix F 2009, im Manual 1110 nicht mehr
geführt"*). **Zwei unabhängige Quellen, dasselbe Ergebnis** — Michigan hat die
Codes aus demselben Grund fallen lassen wie das FBI.

Ebenso wichtig ist die Gegenprobe: Kein Zielcode ist ein Delikt, das nur Bundes-
und Stammesbehörden melden dürfen. Stünde dort eines, wäre die Zeile aus einer
Codeliste abgeschrieben statt aus der Deliktbeschreibung abgeleitet.

### 3.4 Die Zielliste selbst — zweimal abgeschrieben

Abschnitt 3.3 prüft die Crosswalks **gegen** die Codeliste. Damit ist die
Codeliste selbst noch nicht geprüft: Sie ist eine Abschrift aus einem PDF, der
niemand widerspricht. Ein Tippfehler dort verschiebt stillschweigend beide
Städte.

Illinois veröffentlicht mit seinem *NIBRS Information Exchange Package* eine
eigene Abschrift derselben FBI-Liste. Das ist **keine unabhängige Autorität** —
beide stammen aus derselben Spezifikation. Es ist eine zweite Hand, und die
fängt genau das, was ein PDF-Abgleich nicht fängt: unseren eigenen
Übertragungsfehler.

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Die Erwartungen kommen aus den Testdateien und werden hier nicht nachgebaut.
# Zwei Fassungen koennten auseinanderlaufen, und dann zeigte das Notebook eine
# Uebereinstimmung, die der Test nicht mehr prueft.
from tests.config.test_nibrs_offense_codes import IEPD_TABELLE, WORTLAUT  # noqa: E402
import openpyxl                                                           # noqa: E402

_b = openpyxl.load_workbook(IEPD_TABELLE, read_only=True, data_only=True)['OffenseCode']
_zeilen = [z for z in _b.iter_rows(values_only=True)]
_kopf = next(i for i, z in enumerate(_zeilen) if z and z[0] == 'Code')
illinois = {str(z[1]).strip() for z in _zeilen[_kopf + 1:] if z and z[0]}

woertlich = [c for c, z in nibrs_codes.items() if z['nibrs_description'] in illinois]
variante = [c for c in nibrs_codes if c in WORTLAUT and WORTLAUT[c] in illinois]
offen = sorted(set(nibrs_codes) - set(woertlich) - set(variante))

print(f'Delikte bei uns      : {len(nibrs_codes)}')
print(f'Delikte bei Illinois : {len(illinois)}')
print(f'  woertlich gleich   : {len(woertlich)}')
print(f'  Schreibweise erklaert: {len(variante)}')
print(f'  ungeklaert         : {offen or "keines"}')
print()
print('Die erklaerten Abweichungen:')
for c in sorted(WORTLAUT):
    art = 'Umbenennung durch das FBI' if c == '11D' else 'Schreibweise'
    print(f"  {c}  wir: {nibrs_codes[c]['nibrs_description']:<42} IL: {WORTLAUT[c]:<52} {art}")

**66 Bezeichnungen decken sich wörtlich, 15 sind erklärt.** Vierzehn davon sind
Schreibweisen — Kaufmanns-Und, Präfix der Deliktgruppe, Singular gegen Plural.
Die fünfzehnte ist keine: `11D` hiess bis zur Fassung 2024 *Fondling* und heisst
seit der Fassung 2025 *Criminal Sexual Contact*. Das FBI sagt das in seiner
eigenen Änderungsliste; Illinois' Abschrift von 2023.0 führt noch den alten
Namen. Unsere ist die neuere.

Ein Gegentest hält die Ausnahmeliste sauber: Ein Eintrag, dessen Wortlaut
inzwischen übereinstimmt, ist überflüssig und lässt den Test fallen. Ohne das
sammelt eine solche Liste mit der Zeit Ausnahmen an, die keine mehr sind — und
verdeckt damit echte Abweichungen.

## 4. Woher der Crosswalk kommt — und warum er neu gebaut wurde

Die Spalte `match` hält fest, wie jede Zeile zu ihrem NIBRS-Code kam.

**Die erste Fassung dieser Tabelle war fehlerhaft.** Sie entstand aus einem Join
über den I-UCR-Code auf eine Codetabelle der Illinois State Police. Diese Tabelle
ist nach **ILCS-Statut** geschlüsselt und vergibt je Statut ein eigenes
vierstelliges Kürzel; Chicagos IUCR-Katalog benutzt dieselben Kürzel für etwas
anderes. Bei 86 von 240 gemeinsamen Codes beschrieb das Statut ein völlig
anderes Delikt — und von dort kam der NIBRS-Code.

In [ ]:
herkunft = collections.Counter(z['match'] or '(leer)' for z in iucr_map)
gesamt = len(iucr_map)

erklaerung = {
    'nibrs_definition': 'Chicagos Beschreibung hat eine klare NIBRS-Entsprechung',
    'nibrs_auffang': 'NIBRS kennt das Delikt nicht, deshalb 90Z',
    'nibrs_ermessen': 'zwei Codes waren vertretbar, Begruendung steht in note',
    '(leer)': 'kein Delikt, bewusst nicht zugeordnet',
}
print(f"{'Herkunft':<20}{'Zeilen':>8}{'Anteil':>9}   Bedeutung")
for schluessel, n in herkunft.most_common():
    print(f'{schluessel:<20}{n:>8}{n / gesamt:>9.1%}   {erklaerung.get(schluessel, "")}')

print()
print('Die Ermessensfaelle, jeder mit Begruendung:')
for z in iucr_map:
    if z['match'] == 'nibrs_ermessen':
        print(f"  {z['iucr']} -> {z['ibr_code']:<4} {z['note'][:66]}")

**Kein Code stammt mehr aus einer Positionsentsprechung.** Die Zuordnung
entsteht aus Chicagos eigener Deliktbeschreibung gegen die amtliche Codeliste,
festgehalten in [`tools/build_iucr_nibrs_mapping.py`](../tools/build_iucr_nibrs_mapping.py).
Das Skript erzeugt die Datei reproduzierbar; ein zweiter Lauf liefert dieselben
Bytes.

**106 der 434 Zeilen** bekamen dabei einen anderen Code. Die groben Fehler der
alten Fassung:

| IUCR | Delikt | alt | neu |
| --- | --- | --- | --- |
| `1140` | DECEPTIVE PRACTICE / EMBEZZLEMENT | `250` Forgery | **`270` Embezzlement** |
| `1265` | CRIMINAL DAMAGE / LIBRARY VANDALISM | `370` Pornography | **`290` Vandalism** |
| `0451` | BATTERY / AGG. P.O. – OTHER FIREARM | `35A` Drogen | **`13A`** |
| `0263` | CRIMINAL SEXUAL ASSAULT / KNIFE | `36A` Incest | **`11A` Rape** |
| `1512` | PROSTITUTION / SOLICITING | `11A` Rape | **`40B`** |

Besonders die letzte Zeile: vier Prostitutionsdelikte standen auf einem
Vergewaltigungscode und hätten jede Auswertung der Sexualdelikte verzerrt.

**Die Lehre ist allgemeiner als der Fall.** Ein Schlüssel, der in zwei Systemen
dieselbe *Form* hat, ist deshalb noch nicht derselbe Schlüssel. Ein Join darüber
scheitert nicht — er liefert ein formgültiges und falsches Ergebnis. Dagegen
hilft keine Schemaprüfung, sondern nur ein Abgleich der **Bedeutung**: hier der
Deliktbeschreibungen, die beide Seiten mitliefern.

### 4.1 Die Gegenprobe: was eine zweite Quelle dazu sagt

Abschnitt 4 zeigt, warum die ISP-Tabelle **kein Crosswalk** ist. Sie ist damit
aber nicht wertlos — nur an der falschen Stelle eingesetzt.

**Das Problem, das sie löst.** Jede Prüfung in Abschnitt 3 vergleicht die
Zuordnung mit *Chicagos eigener Beschreibung*. Nach diesem Massstab war
`0560 ASSAULT / SIMPLE → 13B Simple Assault` richtig — und trotzdem falsch:
Illinois trennt **Assault** als Drohung ohne Berührung (720 ILCS 5/12-1) von
**Battery** als Berührung (5/12-3). Die NIBRS-Entsprechung der Drohung ist
`13C Intimidation`. Ein Selbstvergleich kann so etwas nicht finden.

**Die Lösung ist ein Tor davor.** Verglichen wird nur dort, wo der ILCS-Wortlaut
und Chicagos Beschreibung dasselbe Delikt meinen — gemessen an der Überlappung
der Inhaltswörter, Schwelle `0.6`. Unterhalb davon ist es Namensähnlichkeit ohne
Aussage.

Die nächste Zelle rechnet das mit **derselben Funktion**, die auch der Test
benutzt. Notebook und Test können dadurch nicht auseinanderlaufen.

In [ ]:
from tests.config.test_iucr_nibrs_mapping import (   # noqa: E402
    ABWEICHUNGEN, DECKUNG, ISP_TABELLE, vergleichbare_zeilen,
)

blatt = openpyxl.load_workbook(ISP_TABELLE, read_only=True, data_only=True)
blatt = blatt[blatt.sheetnames[0]]
isp = {
    str(iucr).strip(): (str(literal or ''), str(ibr).strip())
    for _ilcs, literal, iucr, ibr, _kat in blatt.iter_rows(min_row=4, values_only=True)
    if iucr and ibr
}

gemeinsam = sum(1 for z in iucr_map if z['ibr_code'] and z['iucr'] in isp)
# vergleichbare_zeilen liefert je Treffer (unsere Zeile, ISP-Wortlaut, ISP-Code)
vergleichbar = list(vergleichbare_zeilen(iucr_map, isp))
einig = [z for z, _wortlaut, ibr in vergleichbar if z['ibr_code'] == ibr]
uneinig = {z['iucr']: (z['ibr_code'], ibr)
           for z, _wortlaut, ibr in vergleichbar if z['ibr_code'] != ibr}

print(f'Chicagos Katalog          : {len(iucr_map)} Codes')
print(f'  davon in der ISP-Tabelle: {gemeinsam}')
print(f'  davon Wortlaut deckend  : {len(vergleichbar)}  (Schwelle {DECKUNG})')
print(f'    ISP stimmt zu         : {len(einig)}')
print(f'    ISP widerspricht      : {len(uneinig)}')
print()
print('Jede Abweichung mit ihrer Begruendung:')
for iucr in sorted(uneinig):
    unser, ihrer = uneinig[iucr]
    grund = ABWEICHUNGEN.get(iucr, (None, None, 'KEINE BEGRUENDUNG HINTERLEGT'))[2]
    zeile = next(z for z in iucr_map if z['iucr'] == iucr)
    print(f"  {iucr}  {zeile['secondary_description'][:34]:36} wir {unser:<4} ISP {ihrer:<4}")
    print(f"        {grund}")

unbegruendet = set(uneinig) - set(ABWEICHUNGEN)
print()
print('ohne Begruendung:', sorted(unbegruendet) or 'keine')

**Ein Viertel der Tabelle bekommt ein zweites Augenpaar.** 240 der 434 Codes
stehen auch in der ISP-Tabelle, aber nur bei **111** deckt sich der Wortlaut so
weit, dass ein Vergleich etwas aussagt. Die übrigen 129 teilen sich nur das
Kürzel — dort meint die ISP ein anderes Delikt.

Beim ersten Lauf stimmte die ISP 97 mal zu und widersprach 14 mal. **Sechs dieser
Widersprüche waren unsere Fehler** und sind umgebucht; die Zelle oben zeigt
deshalb heute 103 zu 8:

| IUCR | | vorher | jetzt | Fälle |
| --- | --- | --- | --- | ---: |
| `1185` | Deceptive Collection Practices | `26A` | **`26C`** | 33 |
| `1573` | Crim Sexual Abuse – Penetration | `11A` | **`36B`** | 10 |
| `4740` | Unlawful Use of Body Armor | `520` | **`90Z`** | 1 |
| `1574` | Fornication | `90Z` | **`90C`** | 1 |
| `4810` | Compounding a Crime | `90Z` | **`510`** | 0 |
| `1531` | Juvenile Pimping | `40B` | **`64A`** | 0 |

`1531` beantwortet dabei die Frage, die in Abschnitt 9 als offen stand: **ob
Minderjährigkeit allein Menschenhandel begründet.** Die ISP antwortet
differenziert — *Fördern* einer minderjährigen Prostituierten ist `64A`
(`11-14.4`), *Freier sein* bleibt `40C` (`11-18.1(a)`). `1526` ist damit belegt
statt Ermessen; `1521` folgt `1531` als Schlussfolgerung und bleibt Ermessen,
weil die ISP zu dem Kürzel keine Zeile führt.

**Die acht übrigen sind begründete Abweichungen**, oben ausgedruckt. Sie zerfallen
in zwei Arten: Statut-Artefakte, bei denen die ISP-Zeile schlicht ein anderes
Delikt ist (`0810`, `0820`, `1505`), und Sachfragen, bei denen die
NIBRS-Definition eindeutiger ist als die Wahl der ISP (`1030`, `1145`, `1147`,
`3920`, `4750`).

**Was der Test daraus macht.** `test_abweichungen_von_der_landespolizei_sind_begruendet`
verlangt **nicht** Übereinstimmung — dafür ist die Tabelle zu sehr nach Statut
geschlüsselt. Er verlangt, dass die Menge der Abweichungen genau der
dokumentierten entspricht. Genau das hat bei `0560` gefehlt: Der Code wäre auf
der Liste gelandet, und jemand hätte aufschreiben müssen, warum wir von der
Landespolizei abweichen — und dabei gemerkt, dass er es nicht kann.

Gegenprobe: `0560` zurück auf `13B` gesetzt meldet der Test
`neu abweichend: {'0560': ('13B', '13C')}`.

**Was er nicht leistet.** 111 von 432 Codes, also 26 %, und für Detroit nichts.
In der 13er-Familie hatten nur `0460` und `0560` eine passende ISP-Zeile; ein
falsches `051A` fände er nicht. Kein Netz, sondern ein zweites Augenpaar auf dem
Viertel, das die Landespolizei selbst beurteilt hat.

## 5. Abdeckung: was zugeordnet ist und was nicht

In [ ]:
ohne_zuordnung = [z for z in iucr_map if not z['ibr_code'].strip()]
print(f'ohne NIBRS-Zuordnung: {len(ohne_zuordnung)} von {len(iucr_map)}')
for z in ohne_zuordnung:
    print(f"  {z['iucr']}  {z['primary_description']} / {z['secondary_description']}"
          f"   (aktiv={z['active']})")

if ohne_zuordnung:
    codes = tuple(z['iucr'] for z in ohne_zuordnung)
    treffer = con.execute(
        f'SELECT count(*) FROM {CHICAGO} WHERE iucr IN {codes}'
    ).fetchone()[0]
    print(f'\n  davon in den Falldaten: {treffer}')

in_daten = {z[0] for z in con.execute(
    f'SELECT DISTINCT iucr FROM {CHICAGO}').fetchall()}
bekannt = {z['iucr'] for z in iucr_map}
print(f'\nverschiedene IUCR-Codes in den Falldaten: {len(in_daten)}')
print('nicht in der Mapping-Tabelle:', sorted(in_daten - bekannt) or 'keiner')

**Der Unterschied zwischen zwei Aussagen** ist hier wichtig:

- „Jede Zeile der Tabelle trägt einen NIBRS-Code" — **falsch**, zwei sind offen.
  Beide sind aber bewusst offen: `1481` ist der Entzug einer Waffentragerlaubnis,
  `9901` ein Sammelkennzeichen für häusliche Gewalt. Keines ist ein Delikt.
- „Jeder Code, der in den Falldaten vorkommt, ist zugeordnet" — **richtig**.
- „Jeder Fall trägt ein echtes NIBRS-Delikt" — **falsch**. 2,97 % landen im
  Auffang `90Z`, weil NIBRS das Delikt nicht kennt.

Für H3 zählt die dritte, und sie ist die unbequemste. Wer die zweite für die
dritte hält, behauptet mehr als er gemessen hat.

## 6. Detroits Seite: MICR — und die zweite Codeliste

Michigan meldet über **MICR**, den *Michigan Incident Crime Reporting*-Standard.
Detroits Falldaten führen ihn als `state_offense_code`, daneben eine gröbere
`offense_category`.

**MICR führt zwei Codelisten, nicht eine.** Das Handbuch trennt sie als eigene
Datenfelder:

| Feld | Frage | Beispiel |
| --- | --- | --- |
| *Offense File Class* | Was ist passiert? | `09001` Murder/Non-Negligent Manslaughter |
| *Arrest Charge* (Data Element #46) | Womit wird jemand beschuldigt? | `0901` Family, Gun |

Die Arrest-Charge-Liste ist **unter** den File Classes gegliedert — 833 Arrest
Charges unter 104 File Classes:

```text
09001 – Murder/Non-Negligent Manslaughter     ← File Class, fünfstellig
    Family, Gun                        0901   ← Arrest Charge, vierstellig
    Family, Other Weapon               0902
```

**Detroit veröffentlicht beide in einem vierstelligen Feld.** File Classes sind
fünfstellig und werden gekürzt (`13001` → `1301`), Arrest Charges sind von Haus
aus vierstellig. Dadurch sind **54 Kürzel doppeldeutig**: `0901` ist zugleich die
gekürzte File Class `09001` und der Arrest Charge *Family, Gun*.

Genau daran ist ein erster Extraktionsversuch gescheitert — er lieferte
`0901 → "Willful Killing – Family – Gun"`, also einen Arrest Charge als File
Class. Aufgefallen ist es erst beim Abgleich mit Detroits eigenen
Beschreibungen. Für alle 54 stützt Detroits `offense_description` die
File-Class-Lesart; **keine einzige widerspricht ihr.**

`config/micr_nibrs_mapping.csv` deckt heute **98,10 %** der Detroit-Fälle ab. Die
Spalte `micr_code_type` sagt je Zeile, welche der beiden Listen gemeint ist.

In [ ]:
zahlen = con.execute(f'''
    SELECT count(DISTINCT attributes.offense_category),
           count(DISTINCT attributes.state_offense_code),
           count(DISTINCT trim(attributes.offense_description))
    FROM {DETROIT}
''').fetchone()
print(f'offense_category    {zahlen[0]:>4}   <- Detroits eigene Gliederung, ungenutzt')
print(f'state_offense_code  {zahlen[1]:>4}   <- MICR, feiner, jetzt ueber micr_nibrs_mapping.csv')
print(f'offense_description {zahlen[2]:>4}')

print('\nMICR-Codes je Kategorie, die groessten:')
for kat, codes, n in con.execute(f'''
    SELECT attributes.offense_category, count(DISTINCT attributes.state_offense_code),
           count(*)
    FROM {DETROIT} GROUP BY 1 ORDER BY 2 DESC LIMIT 5
''').fetchall():
    print(f'  {kat[:30]:<30}{codes:>3} Codes  {n:>7,} Faelle')

### 6.1 Zwei Codelisten in einem Feld, nachgerechnet

Die Spalte `micr_code_type` hält fest, welche der beiden Listen eine Zeile
meint. Ohne sie ist ein vierstelliger MICR-Code nicht auflösbar.

In [ ]:
arten = collections.Counter(z['micr_code_type'] or '(leer)' for z in micr_map)
print('Code-Arten in der Tabelle:')
for art, n in arten.most_common():
    print(f'  {art:<16}{n:>4}')

arrest = [z for z in micr_map if z['micr_code_type'] == 'arrest_charge']
angeschlossen = [z for z in arrest if z['mapped'] == 'true']
print(f'\nArrest Charges: {len(arrest)}, davon angeschlossen {len(angeschlossen)}')
print('\nDer Weg Arrest Charge -> File Class -> NIBRS, Stichprobe:')
nach_klasse = {z['micr_file_class']: z for z in micr_map
               if z['micr_code_type'] == 'file_class' and z['micr_file_class']}
for z in angeschlossen[:6]:
    quelle = nach_klasse[z['micr_file_class']]
    print(f"  {z['micr_code']}  ueber File Class {z['micr_file_class']} "
          f"({quelle['micr_description'][:30]:<32}) -> {z['nibrs_code']}")

print('\nBewusst offen geblieben:')
for z in micr_map:
    if z['mapped'] == 'false':
        print(f"  {z['micr_code']}  {z['note'][:96]}")

# Wirkung auf die echten Daten
zug = tuple(z['micr_code'] for z in micr_map if z['mapped'] == 'true')
gesamt, gedeckt = con.execute(f'''
    SELECT count(*), count(*) FILTER (attributes.state_offense_code IN {zug})
    FROM {DETROIT}''').fetchone()
print(f'\nAbdeckung der Detroit-Faelle: {gedeckt:,} von {gesamt:,} = {gedeckt/gesamt:.2%}')

**20 der 23 Arrest Charges sind angeschlossen**, und keine einzige Zuordnung
ist geraten: Die erste Stufe steht im MICR-Handbuch, die zweite in dieser Tabelle
selbst. Jede dieser Zeilen lässt sich nachrechnen — ein Test tut das auch.

Die Abdeckung steigt damit von 94,96 % auf **98,10 %**.

**Drei blieben dabei offen**, weil Detroits eigene Beschreibung dem Handbuch
widerspricht. Bei `3072` — mit 1'447 Fällen der grösste Posten — stand 26A
*False Pretenses* gegen 23C *Shoplifting*. Hier zu raten wäre genau der Fehler
gewesen, der den Chicago-Crosswalk 106 falsche Zeilen gekostet hat; die Notiz
lautete deshalb *nur Detroit kann das auflösen*.

**Detroit hat es aufgelöst**, in zwei Feldern der Quelle, die wir bis dahin
nicht gelesen hatten: `arrest_charge` trägt die fünfstellige File Class,
`charge_description` deren Namen im Klartext. Beide sind über alle 1'647 Zeilen
eindeutig und nennen dieselbe File Class, deren Bezeichnung im Handbuch wörtlich
mit Detroits eigener übereinstimmt. Hergeleitet ist das in
[`11_detroit_source.ipynb`](11_detroit_source.ipynb), Abschnitt 7.1.

Diese drei Zeilen tragen deshalb als Herkunft **`detroit_charge_file_class`**
statt des Handbuchs — der Beleg ist hier die Quelle, nicht das Dokument. Damit
steht Detroit bei **100 %**.

### 6.2 Ist die feinere Ebene brauchbar?

Zwei verschiedene Fragen, die man leicht vermischt:

1. **Trägt ein MICR-Code immer dieselbe Kategorie?** Wenn nicht, ist die
   Zuordnung Code → Kategorie mehrdeutig.
2. **Trägt ein MICR-Code immer dieselbe Beschreibung?** Wenn nicht, ist die
   Erfassung innerhalb einer Kategorie uneinheitlich.

Die Zelle prüft beides getrennt.

In [ ]:
mehrdeutig = con.execute(f'''
    SELECT attributes.state_offense_code AS code,
           count(DISTINCT attributes.offense_category) AS kategorien,
           count(*) AS zeilen
    FROM {DETROIT} WHERE attributes.state_offense_code IS NOT NULL
    GROUP BY 1 HAVING count(DISTINCT attributes.offense_category) > 1
    ORDER BY zeilen DESC
''').fetchall()
print(f'MICR-Codes mit mehr als einer Kategorie: {len(mehrdeutig)} von {zahlen[1]}')
for code, kategorien, zeilen in mehrdeutig:
    print(f'  {code}: {kategorien} Kategorien, {zeilen:,} Zeilen')

print('\nZweite Frage - derselbe Code, verschiedene Beschreibungen (MICR 2301):')
for kat, beschr, n in con.execute(f'''
    SELECT attributes.offense_category, trim(attributes.offense_description), count(*)
    FROM {DETROIT} WHERE attributes.state_offense_code = '2301'
    GROUP BY 1,2 ORDER BY 3 DESC
''').fetchall():
    print(f'  {kat:<10}{beschr[:44]:<46}{n:>6,}')

**Die feinere Ebene ist fast sauber.** Nur zwei von 97 Codes überspannen
Kategorien, und bei den Beschreibungen sind die Abweichungen Einzelfälle. MICR wäre also nutzbar — die
Entscheidung für die gröbere Ebene ist eine Abwägung und keine Notwendigkeit.

Was dagegen spricht, MICR jetzt zu nutzen: Für Chicago müsste dann ebenfalls die
feinere Ebene gelten, also IUCR statt `fbi_code`, und damit stünde die
NIBRS-Tabelle mit ihren zwei Fünfteln Handarbeit im Weg zur Vergleichszahl statt
daneben.

### 6.3 Eine Bereinigungsaufgabe, die hier sichtbar wird

In [ ]:
roh, getrimmt, gepolstert, detroit_zeilen = con.execute(f'''
    SELECT count(DISTINCT attributes.offense_description),
           count(DISTINCT trim(attributes.offense_description)),
           count(*) FILTER (attributes.offense_description
                            <> trim(attributes.offense_description)),
           count(*)
    FROM {DETROIT}
''').fetchone()

print(f'Zeilen mit Randleerzeichen: {gepolstert:,} von {detroit_zeilen:,} '
      f'= {gepolstert / detroit_zeilen:.1%}')
print()
print(f'verschiedene Beschreibungen, ungetrimmt: {roh}')
print(f'verschiedene Beschreibungen, getrimmt  : {getrimmt}')
print(f'-> {roh - getrimmt} Scheinwerte allein durch Leerzeichen')
print()
print('Bronze laesst den Text wie geliefert; das Trimmen gehoert nach Silver.')

**Fast die Hälfte der Beschreibungen ist ein Artefakt.** Ungetrimmt zählt die
Quelle 152 verschiedene Deliktbeschreibungen, getrimmt nur 81 — **71 Werte
entstehen allein durch Leerzeichen**.

Wer ohne `trim()` gruppiert, bekommt dieselbe Beschreibung zweimal und hält den
Unterschied womöglich für eine Erfassungsvariante. Das ist genau die Sorte
Fehler, die in einer Auswertung nicht auffällt, weil das Ergebnis plausibel
aussieht.

Bronze lässt den Text wie geliefert — das ist richtig, denn die Auffüllung
gehört zur Quelle. Das Trimmen gehört nach Silver, zusammen mit dem Parsen der
Zeitstempel und dem Auflösen der Fehlwerte.

## 7. Was das für H3 bedeutet

**H3 lautet:** Die Deliktschlüssel beider Städte lassen sich vollständig auf
NIBRS abbilden.

Bis `silver/crime_offense` gebaut war, liess sich das nicht beantworten — die
Crosswalks lagen als Referenzdateien vor, ohne auf die Falldaten angewandt zu
sein. Jetzt ist es eine Messung. Die nächste Zelle rechnet sie.

In [ ]:
if OFFENSE is None:
    print('silver/crime_offense fehlt - zuerst build_crime_offense laufen lassen')
else:
    print(f"{'Stadt':<10}{'Delikte':>9}{'zugeordnet':>12}{'davon 90Z':>11}"
          f"{'echtes Delikt':>15}{'gar nicht':>11}")
    summe = collections.Counter()
    for stadt, g, zug, auffang, ohne in con.execute(f'''
            SELECT city, count(*), count(*) FILTER (nibrs_mapped),
                   count(*) FILTER (nibrs_code = '90Z'),
                   count(*) FILTER (NOT nibrs_mapped)
            FROM {OFFENSE} GROUP BY city ORDER BY city''').fetchall():
        echt = zug - auffang
        print(f'{stadt:<10}{g:>9,}{zug/g:>11.1%}{auffang:>11,}{echt/g:>14.1%}{ohne:>11,}')
        for k, v in (('g', g), ('zug', zug), ('auffang', auffang), ('ohne', ohne)):
            summe[k] += v
    g, zug, auffang, ohne = summe['g'], summe['zug'], summe['auffang'], summe['ohne']
    print(f"{'zusammen':<10}{g:>9,}{zug/g:>11.1%}{auffang:>11,}"
          f"{(zug-auffang)/g:>14.1%}{ohne:>11,}")

    print('\nDelikte, die in beiden Staedten vorkommen - dieselbe Kategorie, '
          'erstmals dieselbe Bedeutung:')
    print(f"  {'Code':<6}{'Delikt':<40}{'Chicago':>9}{'Detroit':>9}")
    for c, t, chi, det in con.execute(f'''
            SELECT nibrs_code, any_value(nibrs_description),
                   count(*) FILTER (city = 'chicago'),
                   count(*) FILTER (city = 'detroit')
            FROM {OFFENSE} WHERE nibrs_code IS NOT NULL GROUP BY nibrs_code
            HAVING count(*) FILTER (city = 'chicago') > 0
               AND count(*) FILTER (city = 'detroit') > 0
            ORDER BY count(*) DESC LIMIT 8''').fetchall():
        print(f'  {c:<6}{t[:38]:<40}{chi:>9,}{det:>9,}')

**H3 ist teilweise falsifiziert** — nicht weil etwas fehlt, sondern weil
„vollständig" zweierlei heissen kann:

- **Abdeckung ja, und zwar ganz.** Jeder Quellcode beider Städte ist bekannt.
  Nichts fällt stillschweigend heraus; das war die eigentliche Sorge.
- **Abbildung nein.** Nur 96,8 % der Delikte tragen ein *echtes* NIBRS-Delikt.
  10'994 landen auf `90Z All Other Offenses` — gültig, für einen Deliktvergleich
  aber ohne Aussage: Registrierungspflichten, Bewährungsauflagen,
  Illinois-Eigenheiten wie *Criminal Fortification*.

Ein Detail, das die Zahl relativiert: Im **Katalog** sind 69 der 434
Chicago-Zeilen Auffangzeilen (15,9 %), in den **Falldaten** nur 2,97 %. Die
Delikte ohne NIBRS-Entsprechung sind überwiegend seltene. Gut für den Vergleich
— aber gemessen und nicht angenommen.

**Detroits Lücke ist seit dem Anschluss von `3072`, `5421` und `5422` ganz
geschlossen** (Abschnitt 6.1). Offen bleiben genau **drei Chicagoer Zeilen** —
`1481` Widerruf einer Waffentragbewilligung und `9901`, eine Sammelzeile ohne
Delikt. Sie sind bewusst nicht zugeordnet, weil sie keine Tat beschreiben.

Die Falsifikation von H3 hängt damit fast nur noch am Auffang `90Z` — an der
Abbildung, nicht an der Abdeckung.

**Und die Harmonisierung läuft jetzt tatsächlich über NIBRS**, nicht mehr über
`fbi_code` (SRS) und `offense_category`. Die Zutaten aus Abschnitt 3 sind
angewandt, nicht nur vorhanden. Absolut vergleichbar sind die Zahlen damit noch
nicht — Chicago hat rund viermal so viele Einwohner. Dafür braucht es den Nenner
aus `silver/tract`, und das gehört nach Gold.

## 8. Quellenverweise

**Standard und Herausgeber**

| Was | Adresse |
| --- | --- |
| NIBRS beim Bureau of Justice Statistics | <https://bjs.ojp.gov/national-incident-based-reporting-system-nibrs> |
| NIBRS beim FBI | <https://www.fbi.gov/how-we-can-help-you/more-fbi-services-and-information/ucr/nibrs> |
| *A Guide to Understanding NIBRS* (FBI, PDF) | <https://www.fbi.gov/file-repository/ucr/a-guide-to-understanding-nibrs.pdf> |
| CRS-Bericht R46668, mit Gegenüberstellung der SRS- und NIBRS-Deliktlisten | <https://www.congress.gov/crs-product/R46668> |

**Illinois und Chicago**

| Was | Adresse |
| --- | --- |
| IUCR-Codes, Datensatz des Chicago Data Portal (`c7ck-438e`) | <https://data.cityofchicago.org/Public-Safety/Chicago-Police-Department-Illinois-Uniform-Crime-R/c7ck-438e> |
| Crime Reporting der Illinois State Police | <https://isp.illinois.gov/CrimeReporting/NIBRS> |
| Illinois-UCR-Portal | <https://ilucr.nibrs.com/> |

**Michigan und Detroit**

| Was | Adresse |
| --- | --- |
| MICR beim Michigan State Police | <https://www.michigan.gov/msp/divisions/cjic/micr> |

**Im Repository — die Belege**

| Was | Wo |
| --- | --- |
| Amtliche NIBRS-Codeliste, 81 Codes | `config/nibrs_offense_codes_2025.csv` |
| Quelle dazu: *NIBRS Technical Specification 2025*, Anhang A | `docs/nibrs_technical_specification_2025.pdf` |
| IUCR nach NIBRS | `config/iucr_nibrs_mapping.csv`, erzeugt von [`tools/build_iucr_nibrs_mapping.py`](../tools/build_iucr_nibrs_mapping.py) |
| MICR nach NIBRS | `config/micr_nibrs_mapping.csv`, Arrest Charges angeschlossen von [`tools/extend_micr_arrest_charges.py`](../tools/extend_micr_arrest_charges.py) |
| Quellen dazu | `docs/micr_manual_1110_2024.pdf`, `docs/micr_appendix_f_2009.pdf` |
| Codetabelle der ISP, Fassung 2026 | `docs/isp_nibrs_code_table_2026.xlsx` — die Gegenprobe aus 4.1 |
| Illinois' zweite Abschrift der FBI-Codeliste | `docs/illinois_nibrs_iepd_2023.xlsx`, Blatt `OffenseCode` — der Abgleich aus 3.4 |
| Die Crosswalks angewandt | `src/crime/silver_pipeline/build_crime_offense.py` → `data/silver/dataset=crime_offense/` |
| Die Prüfungen dazu | [`tests/config/test_iucr_nibrs_mapping.py`](../tests/config/test_iucr_nibrs_mapping.py), [`test_nibrs_offense_codes.py`](../tests/config/test_nibrs_offense_codes.py), [`test_micr_nibrs_mapping.py`](../tests/config/test_micr_nibrs_mapping.py) |
| Abgabetext, Kapitel 3.3, 8.2 und 8.5 | [`docs/ln1_konzept.md`](../docs/ln1_konzept.md) |
| Was daraus an Auswertung entstand | [`reports/`](../reports/) — Juni/November-Analyse, Artans Vergleich, der Abgleich beider |

**Zur ISP-Codetabelle im Repository:** Sie ist **kein Crosswalk** und wird auch
nicht als einer benutzt. Sie ist nach ILCS-Statut geschlüsselt; ein Join über den
I-UCR-Code auf Chicagos Katalog liefert ein formgültiges und falsches Ergebnis —
daran ist die erste Fassung gescheitert (Abschnitt 4). Als **Gegenprobe** trägt
sie trotzdem, sobald ein semantisches Tor davor steht: Abschnitt 4.1 vergleicht
nur, wo sich der Wortlaut deckt. Das ist ein Viertel der Tabelle, und genau dort
lag `0560`.

**Zu Illinois' IEPD-Arbeitsmappe:** Gebraucht wird daraus nur das Blatt
`OffenseCode`. Ihr Blatt `OffenseILUCRCode` ist die ältere Ausgabe (Fassung
2023.0) derselben Tabelle, die oben als Fassung 2026 liegt — wer die Gegenprobe
nachrechnet, nimmt die neuere.

**Zu den Michigan-Dokumenten:** Sie liegen als PDF im Repository statt nur als
Verweis, weil `michigan.gov` automatisierte Abrufe mit HTTP 403 beantwortet.

**Stand der Abrufe:** Die Angaben zu NIBRS und zum Chicago Data Portal wurden am
11.9.2026 geprüft; der IUCR-Katalog wird von `tools/build_iucr_nibrs_mapping.py`
bei jedem Lauf frisch geladen und gegen die Datei abgeglichen. Die Seite des FBI
antwortete auf automatisierte Abrufe mit HTTP 403; sie ist im Browser erreichbar
und hier als Verweis aufgeführt, nicht als geprüfte Quelle.

## 9. Offen

- **Drei Detroit-Codes hängen an der Quelle statt am Handbuch.** `3072`, `5421`
  und `5422` sind über Detroits eigene Felder zugeordnet (Abschnitt 6.1), nicht
  über das MICR-Handbuch, das ihnen etwas anderes zuschreibt. Der Beleg ist
  stark — zwei unabhängige Spalten, alle 1'647 Zeilen —, aber er ist eine
  Rechnung und kein Dokument. Eine Bestätigung durch Detroit stünde noch aus.
- **Vier MICR-Zielcodes sind zurückgezogen** (`90A`, `90E`, `90H`, `90I`). Sie
  bleiben im Crosswalk, weil die MICR-Quellen sie führen, und tragen eine Notiz.
  `7070 Runaway` ist über die File Class auf `90I` angeschlossen und erbt das.
- **`1521` Keeping a Place of Juvenile Prostitution** bleibt Ermessen. Die Frage,
  ob Minderjährigkeit allein `64A` begründet, hat die ISP-Tabelle für `1531` und
  `1526` beantwortet (Abschnitt 4.1) — zu `1521` führt sie keine Zeile. Die
  Zuordnung folgt `1531`, weil Illinois beide 2013 in `11-14.4` zusammenfasste;
  das ist eine Schlussfolgerung und kein Beleg.
- **Für Detroit gibt es keine Gegenprobe.** Michigan veröffentlicht keine
  Entsprechung zur ILCS-Tabelle der ISP. Der MICR-Crosswalk ist gegen zwei
  Handbücher belegt, aber nicht gegen eine zweite Zuordnung.
- **Die Gegenprobe deckt ein Viertel der Chicagoer Tabelle**, und in der
  13er-Familie nur `0460` und `0560`. Die grossen Alltagsdelikte prüft weiter
  allein der Abgleich mit Chicagos eigener Beschreibung.
- **Die Rate je Einwohner** — absolute Fallzahlen sind zwischen Städten mit
  unterschiedlicher Einwohnerzahl nicht vergleichbar. Der Nenner steht in
  `silver/tract` bereit; die Verbindung gehört nach Gold.

In [ ]:
con.close()
print('Verbindung geschlossen.')